# 01 - Data profiling and cleaning\nAll cleaning logic lives in this notebook (no hidden modules). Goal: profile the 12 raw CSVs, fix the issues in PRD 19.1, log every transformation, validate integrity, and build the order-level base table.\n\n**Principle:** test a rule on the data before applying it (e.g. are negative values sign flips or real errors?) and write every decision to `data/cleaned/cleaning_log.csv`.\n\nOutputs: `data/cleaned/*.csv` (12 tables), `cleaning_log.csv`, `validation_report.csv`, `order_base.csv`.

In [1]:
# ---- Setup -------------------------------------------------------------------------------------------
import re, difflib, pathlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 40)

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
RAW, CLEAN = ROOT / "data" / "raw", ROOT / "data" / "cleaned"
CLEAN.mkdir(parents=True, exist_ok=True)
TABLES = ["cities", "customers", "restaurants", "menu", "delivery_partners", "promotions",
          "orders", "order_items", "payments", "customer_feedback", "weather", "traffic"]

# Read EVERYTHING as text first: bad values ("abc" phone numbers, mixed dates) must survive loading.
raw = {t: pd.read_csv(RAW / f"{t}.csv", dtype=str) for t in TABLES}
print({t: d.shape for t, d in raw.items()})

{'cities': (25, 5), 'customers': (12180, 13), 'restaurants': (1200, 13), 'menu': (8997, 8), 'delivery_partners': (2000, 10), 'promotions': (300, 6), 'orders': (20705, 15), 'order_items': (42378, 6), 'payments': (20493, 6), 'customer_feedback': (14200, 7), 'weather': (18264, 7), 'traffic': (18335, 6)}


In [2]:
# ---- Helpers: logging + text cleaning + mixed-format date parsing ------------------------------------
log = []                      # every transformation is recorded here -> data/cleaned/cleaning_log.csv
def note(table, issue, column, n, action):
    n = int(n)
    if n: log.append(dict(table=table, issue=issue, column=column, rows_affected=n, action=action))

def clean_text(s):
    """strip, collapse inner whitespace, blank-like strings -> NaN"""
    s = s.astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    return s.mask(s.isin(["", "nan", "NaN", "None"]))

def to_num(df, cols):
    for c in cols: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def abs_fix(df, table, col):
    """Negative values are SIGN FLIPS (proved above), so abs() rather than dropping the row."""
    note(table, "negative value (sign flip)", col, (df[col] < 0).sum(), "abs() - verified sign error")
    df[col] = df[col].abs(); return df

def parse_mixed_dates(s):
    """Handles YYYY-MM-DD, YYYY.MM.DD, DD-MM-YYYY, 'DD Mon YYYY' and slash dates that mix DD/MM and MM/DD.
    Returns (best_guess, ambiguous_flag, alternative_reading)."""
    s = s.astype("string").str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]"); alt = out.copy()
    for pat, fmt in [(r"^\d{4}-\d{2}-\d{2}$", "%Y-%m-%d"), (r"^\d{4}\.\d{2}\.\d{2}$", "%Y.%m.%d"),
                     (r"^\d{2}-\d{2}-\d{4}$", "%d-%m-%Y"), (r"^\d{1,2} [A-Za-z]{3} \d{4}$", "%d %b %Y")]:
        m = s.str.match(pat, na=False); out[m] = pd.to_datetime(s[m], format=fmt, errors="coerce")
    sl = s.str.match(r"^\d{2}/\d{2}/\d{4}$", na=False)
    a, b = pd.to_numeric(s.str.slice(0, 2), errors="coerce"), pd.to_numeric(s.str.slice(3, 5), errors="coerce")
    dmy = pd.to_datetime(s.where(sl), format="%d/%m/%Y", errors="coerce"); mdy = pd.to_datetime(s.where(sl), format="%m/%d/%Y", errors="coerce")
    out[sl & (a > 12)] = dmy[sl & (a > 12)]; out[sl & (b > 12)] = mdy[sl & (b > 12)]
    amb = sl & (a <= 12) & (b <= 12)
    out[amb] = dmy[amb]; alt[amb] = mdy[amb]          # default DD/MM (Indian convention)
    return out, amb.fillna(False), alt

CITY_ALIASES = {"bangalore": "Bengaluru", "bombay": "Mumbai", "calcutta": "Kolkata", "baroda": "Vadodara", "new delhi": "Delhi",
                "vizag": "Visakhapatnam", "cochin": "Kochi", "poona": "Pune", "madras": "Chennai", "gurgaon": "Delhi"}
def canon_city(s, table, canon):
    """Whitespace/case/alias/typo fixes -> canonical names from cities.csv"""
    r = clean_text(s); key = r.str.lower(); lc = {c.lower(): c for c in canon}
    out = key.map(lc).fillna(key.map(CITY_ALIASES))
    for v in key[out.isna() & r.notna()].unique():
        m = difflib.get_close_matches(v, list(lc), n=1, cutoff=0.8)
        if m: out[key == v] = lc[m[0]]
    note(table, "city spelling/case/whitespace", "City", (out.fillna("") != s.astype("string").fillna("")).sum(), "canonicalised to cities.City")
    return out.astype("string")
print("helpers ready")

helpers ready


## 1. Profile the raw data

In [3]:
# ---- Profile: nulls and exact duplicates per table ---------------------------------------------------
rows = []
for t, d in raw.items():
    nn = d.isna().sum()
    rows.append(dict(table=t, rows=len(d), exact_duplicates=int(d.duplicated().sum()), columns_with_nulls=", ".join(f"{c}({n})" for c, n in nn[nn > 0].items())))
pd.DataFrame(rows)

,table,rows,exact_duplicates,columns_with_nulls
0,cities,25,0,AverageIncome(2)
1,customers,12180,180,"Age(170), Gender(122), Phone(144), Email(332), State(186), Membership(120), PreferredCuisine(245)"
2,restaurants,1200,0,"Cuisine(12), Rating(22)"
3,menu,8997,0,Calories(132)
4,delivery_partners,2000,0,Rating(34)
5,promotions,300,0,
6,orders,20705,205,"CouponCode(13421), FinalAmount(327)"
7,order_items,42378,0,TotalPrice(435)
8,payments,20493,0,TransactionID(183)
9,customer_feedback,14200,0,"CustomerRating(294), Review(1058)"


## 2. Test the assumptions before cleaning\nIf negative `FoodCost` is just a sign error, then `abs(FoodCost) + DeliveryFee - Discount + GST` must reproduce `FinalAmount`.

In [4]:
# ---- Evidence: are negatives real or sign flips? ------------------------------------------------------
o = raw["orders"].drop_duplicates().copy()
for c in ["FoodCost", "DeliveryFee", "Discount", "GST", "FinalAmount"]: o[c] = pd.to_numeric(o[c], errors="coerce")
neg = o[o.FoodCost < 0]
recon = neg.FoodCost.abs() + neg.DeliveryFee - neg.Discount + neg.GST
print(f"orders with negative FoodCost: {len(neg)} | |FoodCost|+Fee-Discount+GST equals FinalAmount for {int(((recon - neg.FinalAmount).abs() < 1e-6).sum())} of {int(neg.FinalAmount.notna().sum())}")
ok = o[o.FoodCost > 0]
print("formula holds on all normal rows:", bool(((ok.FoodCost + ok.DeliveryFee - ok.Discount + ok.GST - ok.FinalAmount).abs() < 1e-6).all()))

orders with negative FoodCost: 181 | |FoodCost|+Fee-Discount+GST equals FinalAmount for 178 of 178
formula holds on all normal rows: False


Slash dates mix DD/MM and MM/DD *inside the same column*; a part > 12 settles it, the rest are ambiguous.

In [5]:
# ---- Evidence: date formats ---------------------------------------------------------------------------
def pattern(x): return re.sub(r"[A-Za-z]", "a", re.sub(r"\d", "9", str(x)))
print(raw["orders"].OrderDate.map(pattern).value_counts().to_string())
dt, amb, alt = parse_mixed_dates(raw["orders"].OrderDate)
print(f"\nslash dates that are ambiguous (both parts <= 12): {int(amb.sum())} of {len(dt)} ({amb.mean():.1%})")

OrderDate
99/99/9999    8218
9999.99.99    4191
99-99-9999    4187
9999-99-99    4109

slash dates that are ambiguous (both parts <= 12): 3222 of 20705 (15.6%)


## 3. Clean table by table (in foreign-key order)

In [6]:
# ---- 1. cities -----------------------------------------------------------------------------------------
C = {}                                              # cleaned tables
d = to_num(raw["cities"].copy(), ["CityID", "Population", "AverageIncome"])
d["City"] = clean_text(d["City"]).str.title()
note("cities", "missing", "AverageIncome", d.AverageIncome.isna().sum(), "imputed with region median")
d["AverageIncome"] = d["AverageIncome"].fillna(d.groupby("Region")["AverageIncome"].transform("median"))
C["cities"] = d; CITIES_LIST = d.City.tolist(); d.head(3)

,CityID,City,Population,Region,AverageIncome
0,1,Mumbai,20398296,West,1435819.0
1,2,Delhi,32939024,North,1373024.0
2,3,Bengaluru,13187628,South,1609572.0


In [7]:
# ---- 2. customers --------------------------------------------------------------------------------------
d = raw["customers"].copy()
note("customers", "duplicate rows", "all", d.duplicated().sum(), "dropped duplicates"); d = d.drop_duplicates().drop_duplicates("CustomerID")
for c in ["Name", "Email", "Gender", "State", "Membership", "PreferredCuisine"]: d[c] = clean_text(d[c])
d["City"] = canon_city(d["City"], "customers", CITIES_LIST)
d = to_num(d, ["CustomerID", "Age", "TotalOrders"])
d = abs_fix(d, "customers", "Age")
bad = (d.Age > 100) | (d.Age < 10); note("customers", "implausible", "Age", bad.sum(), "set NaN then imputed"); d.loc[bad, "Age"] = np.nan
note("customers", "missing", "Age", d.Age.isna().sum(), "imputed with median"); d["Age"] = d.Age.fillna(d.Age.median()).round().astype(int)
ph = d["Phone"].astype("string").str.replace(r"\.0$", "", regex=True).str.strip(); ok = ph.str.fullmatch(r"\d{10}").fillna(False)
note("customers", "invalid/missing phone", "Phone", (~ok).sum(), "set to NaN (not guessable)"); d["Phone"] = ph.where(ok)
pc = d["Pincode"].astype("string").str.replace(r"\.0$", "", regex=True).str.lstrip("-").str.strip(); ok = pc.str.fullmatch(r"\d{6}").fillna(False)
note("customers", "malformed pincode", "Pincode", (~ok).sum(), "set to NaN"); d["Pincode"] = pc.where(ok)
for c in ["Gender", "Membership", "PreferredCuisine", "State"]:
    note("customers", "missing", c, d[c].isna().sum(), "filled 'Unknown'"); d[c] = d[c].fillna("Unknown")
note("customers", "missing", "Email", d.Email.isna().sum(), "left NULL (nullable)")
dt, amb, _ = parse_mixed_dates(d["RegistrationDate"]); note("customers", "ambiguous slash date", "RegistrationDate", amb.sum(), "defaulted DD/MM")
d["RegistrationDate"] = dt; d["TotalOrders"] = d["TotalOrders"].fillna(0).clip(lower=0).astype(int)
C["customers"] = d; d.head(3)

,CustomerID,Name,Age,Gender,Phone,Email,City,State,Pincode,RegistrationDate,Membership,TotalOrders,PreferredCuisine
0,1,Aryan Maharaj,35,Male,1960013389,issaconi@example.org,Nagpur,Haryana,769177,2022-06-11,Basic,4,Fast Food
1,2,Arunima Ahuja,29,Male,2351161559,caleb78@example.org,Hyderabad,Tripura,776564,2024-02-18,Zomato Pro,6,Mughlai
2,3,Farhan Memon,26,Other,0341316475,kamalapant@example.org,Vadodara,Uttarakhand,507888,2023-05-17,Gold,5,Mughlai


In [8]:
# ---- 3. restaurants and 4. menu ------------------------------------------------------------------------
d = raw["restaurants"].copy().drop_duplicates("RestaurantID")
for c in ["RestaurantName", "Cuisine", "Area", "OwnerName", "RestaurantType"]: d[c] = clean_text(d[c])
d["City"] = canon_city(d["City"], "restaurants", CITIES_LIST)
d = to_num(d, ["RestaurantID", "Rating", "AverageCost", "Latitude", "Longitude"])
note("restaurants", "missing", "Cuisine", d.Cuisine.isna().sum(), "filled 'Unknown'"); d["Cuisine"] = d.Cuisine.fillna("Unknown")
d = abs_fix(d, "restaurants", "AverageCost")
bad = d.Rating > 5; note("restaurants", "rating out of 1-5", "Rating", bad.sum(), "set NaN then imputed"); d.loc[bad, "Rating"] = np.nan
note("restaurants", "missing", "Rating", d.Rating.isna().sum(), "imputed with cuisine median")
d["Rating"] = d.Rating.fillna(d.groupby("Cuisine")["Rating"].transform("median")).fillna(d.Rating.median()).round(1)
C["restaurants"] = d

m = raw["menu"].copy().drop_duplicates("FoodItemID"); m = to_num(m, ["FoodItemID", "RestaurantID", "Price", "PreparationTime", "Calories"])
m["FoodName"], m["Category"] = clean_text(m.FoodName), clean_text(m.Category)
m = abs_fix(m, "menu", "Price")
note("menu", "missing", "Calories", m.Calories.isna().sum(), "imputed with category median")
m["Calories"] = m.Calories.fillna(m.groupby("Category")["Calories"].transform("median")).round()
keep = m.RestaurantID.isin(d.RestaurantID); note("menu", "orphan FK", "RestaurantID", (~keep).sum(), "dropped")
C["menu"] = m[keep]; print(C["restaurants"].shape, C["menu"].shape)

(1200, 13) (8997, 8)


In [9]:
# ---- 5. delivery_partners and 6. promotions -------------------------------------------------------------
d = raw["delivery_partners"].copy().drop_duplicates("DeliveryPartnerID")
d = to_num(d, ["DeliveryPartnerID", "Age", "Rating", "CompletedDeliveries", "AverageDeliveryTime"])
d["Name"], d["VehicleType"] = clean_text(d.Name), clean_text(d.VehicleType); d["City"] = canon_city(d["City"], "delivery_partners", CITIES_LIST)
d = abs_fix(d, "delivery_partners", "AverageDeliveryTime")
note("delivery_partners", "missing", "Rating", d.Rating.isna().sum(), "imputed with median"); d["Rating"] = d.Rating.fillna(d.Rating.median())
d["JoiningDate"], _, _ = parse_mixed_dates(d["JoiningDate"])
q1, q3 = np.percentile(d.AverageDeliveryTime, [25, 75]); iqr = q3 - q1            # vectorised IQR rule (NumPy)
note("delivery_partners", "extreme outlier (IQR*3)", "AverageDeliveryTime", ((d.AverageDeliveryTime > q3 + 3 * iqr) | (d.AverageDeliveryTime < q1 - 3 * iqr)).sum(), "capped at bound")
d["AverageDeliveryTime"] = d.AverageDeliveryTime.clip(q1 - 3 * iqr, q3 + 3 * iqr); C["delivery_partners"] = d

p = raw["promotions"].copy(); p["CouponCode"] = clean_text(p.CouponCode).str.upper(); p["CampaignName"] = clean_text(p.CampaignName)
p = to_num(p, ["PromotionID", "DiscountPercentage"]); sd, _, _ = parse_mixed_dates(p.StartDate); ed, _, _ = parse_mixed_dates(p.EndDate)
sw = ed < sd; note("promotions", "EndDate before StartDate", "StartDate/EndDate", sw.sum(), "swapped")
p["StartDate"], p["EndDate"] = sd.where(~sw, ed), ed.where(~sw, sd); C["promotions"] = p.drop_duplicates("CouponCode")
print(C["delivery_partners"].shape, C["promotions"].shape)

(2000, 10) (300, 6)


In [10]:
# ---- 7. weather and 8. traffic --------------------------------------------------------------------------
w = to_num(raw["weather"].copy(), ["WeatherID", "Temperature", "Rainfall", "Humidity"]).sort_values("WeatherID").reset_index(drop=True)
city = canon_city(w["City"], "weather", CITIES_LIST); n0 = city.isna().sum()
city = city.where(~(city.isna() & (city.ffill() == city.bfill())), city.ffill())      # rows are stored in contiguous per-city blocks
note("weather", "missing", "City", n0 - city.isna().sum(), "recovered from neighbouring rows (per-city blocks)"); w["City"] = city.ffill()
w["Date"], amb, _ = parse_mixed_dates(w["Date"]); note("weather", "ambiguous slash date", "Date", amb.sum(), "defaulted DD/MM (unresolvable: dates are sampled with repeats)")
w["WeatherCondition"] = clean_text(w.WeatherCondition); w = abs_fix(w, "weather", "Humidity"); w["Humidity"] = w.Humidity.clip(upper=100)
hi = np.percentile(w.Rainfall.dropna(), 99); note("weather", "missing", "Rainfall", w.Rainfall.isna().sum(), "imputed with weather-condition median")
w["Rainfall"] = w.Rainfall.fillna(w.groupby("WeatherCondition")["Rainfall"].transform("median"))
note("weather", "extreme outlier", "Rainfall", (w.Rainfall > hi).sum(), f"capped at P99={hi:.0f}mm"); w["Rainfall"] = w.Rainfall.clip(upper=hi)
t_hi = np.percentile(w.Temperature, 99.5); note("weather", "extreme outlier", "Temperature", (w.Temperature > t_hi).sum(), f"capped at P99.5={t_hi:.1f}C"); w["Temperature"] = w.Temperature.clip(upper=t_hi)
C["weather"] = w          # NOTE: (City, Date) is NOT unique - dates repeat - so we aggregate when joining

t = to_num(raw["traffic"].copy(), ["TrafficID", "AverageSpeed"]).sort_values("TrafficID").reset_index(drop=True)
t["City"] = canon_city(t["City"], "traffic", CITIES_LIST); t["Date"], amb, _ = parse_mixed_dates(t["Date"]); note("traffic", "ambiguous slash date", "Date", amb.sum(), "defaulted DD/MM")
t = abs_fix(t, "traffic", "AverageSpeed"); hi = np.percentile(t.AverageSpeed, 99)
note("traffic", "extreme outlier", "AverageSpeed", (t.AverageSpeed > hi).sum(), f"capped at P99={hi:.0f}km/h"); t["AverageSpeed"] = t.AverageSpeed.clip(upper=hi)
t["TrafficLevel"] = clean_text(t.TrafficLevel); med = t.groupby("TrafficLevel")["AverageSpeed"].median(); miss = t.TrafficLevel.isna()
t.loc[miss, "TrafficLevel"] = t.loc[miss, "AverageSpeed"].apply(lambda v: (med - v).abs().idxmin())      # nearest class by speed
note("traffic", "missing", "TrafficLevel", miss.sum(), "inferred from AverageSpeed (nearest class median)")
t["Time"] = t.Time.astype("string").str.slice(0, 5); C["traffic"] = t
print(C["weather"].shape, C["traffic"].shape); med

(18264, 7) (18335, 6)


TrafficLevel
High        16.3
Low         37.9
Moderate    27.2
Severe       8.0
Name: AverageSpeed, dtype: float64

In [11]:
# ---- 9. orders (the core table) -----------------------------------------------------------------------
d = raw["orders"].copy(); note("orders", "duplicate rows", "all", d.duplicated().sum(), "dropped duplicates")
d = d.drop_duplicates().drop_duplicates("OrderID")
d = to_num(d, ["OrderID", "CustomerID", "RestaurantID", "DeliveryPartnerID", "DeliveryTimeMinutes", "FoodCost", "DeliveryFee", "Discount", "GST", "FinalAmount"])
d["OrderStatus"], d["PaymentMethod"] = clean_text(d.OrderStatus), clean_text(d.PaymentMethod); d["CouponCode"] = clean_text(d.CouponCode).str.upper()

# Dates: resolve ambiguous DD/MM vs MM/DD with the payment date of the SAME order (payments are same-day)
dt, amb, alt = parse_mixed_dates(d.OrderDate)
pay = raw["payments"].drop_duplicates("OrderID"); pdt, pamb, _ = parse_mixed_dates(pay.PaymentDate)
ref = pd.Series(pdt.where(~pamb).values, index=pd.to_numeric(pay.OrderID)); r = d.OrderID.map(ref)
flip = amb & r.notna() & alt.notna() & ((alt - r).abs() < (dt - r).abs())
note("orders", "ambiguous slash date", "OrderDate", amb.sum(), "resolved via payment date, else DD/MM"); note("orders", "ambiguous date flipped", "OrderDate", flip.sum(), "MM/DD chosen (closer to payment date)")
d["OrderDate"] = dt.where(~flip, alt); d["OrderTime"] = d.OrderTime.astype("string").str.slice(0, 8)

for c in ["FoodCost", "DeliveryTimeMinutes"]: d = abs_fix(d, "orders", c)
z = d.DeliveryTimeMinutes <= 0; note("orders", "zero delivery time", "DeliveryTimeMinutes", z.sum(), "set NaN"); d.loc[z, "DeliveryTimeMinutes"] = np.nan
out = d.DeliveryTimeMinutes > 180; note("orders", "unrealistic delivery time (>180 min)", "DeliveryTimeMinutes", out.sum(), "set NaN (excluded from modelling)"); d.loc[out, "DeliveryTimeMinutes"] = np.nan
calc = d.FoodCost + d.DeliveryFee - d.Discount + d.GST
note("orders", "missing", "FinalAmount", d.FinalAmount.isna().sum(), "recomputed = FoodCost+Fee-Discount+GST"); d["FinalAmount"] = d.FinalAmount.fillna(calc)

keep = d.CustomerID.isin(C["customers"].CustomerID) & d.RestaurantID.isin(C["restaurants"].RestaurantID) & d.DeliveryPartnerID.isin(C["delivery_partners"].DeliveryPartnerID)
note("orders", "orphan FK (customer/restaurant/partner)", "CustomerID,RestaurantID", (~keep).sum(), "dropped (cannot be attributed)"); d = d[keep]
bad = d.CouponCode.notna() & ~d.CouponCode.isin(C["promotions"].CouponCode); note("orders", "unknown coupon", "CouponCode", bad.sum(), "set NULL"); d.loc[bad, "CouponCode"] = np.nan
C["orders"] = d; print(d.shape, d.OrderDate.min().date(), "->", d.OrderDate.max().date()); d.head(3)

(20475, 15) 2023-01-01 -> 2024-12-31


,OrderID,CustomerID,RestaurantID,DeliveryPartnerID,OrderDate,OrderTime,DeliveryTimeMinutes,FoodCost,DeliveryFee,Discount,CouponCode,GST,FinalAmount,OrderStatus,PaymentMethod
0,100001,7095,474,121,2023-06-17,11:51:00,43.0,206,40,103.0,ZOMKXZZW,10.30,153.30,Delivered,Cash on Delivery
1,100002,9498,828,1776,2023-07-01,10:18:00,18.0,425,20,127.5,ZOM72K4N,21.25,338.75,Food Not Delivered,Debit Card
2,100003,11040,105,27,2023-05-09,19:34:00,50.0,287,30,28.7,ZOMQJT3W,14.35,302.65,Cancelled,Net Banking


In [12]:
# ---- 10-12. order_items, payments, customer_feedback ----------------------------------------------------
oi = to_num(raw["order_items"].copy().drop_duplicates("OrderItemID"), ["OrderItemID", "OrderID", "FoodItemID", "Quantity", "UnitPrice", "TotalPrice"])
oi = abs_fix(oi, "order_items", "Quantity"); note("order_items", "missing", "TotalPrice", oi.TotalPrice.isna().sum(), "recomputed Quantity*UnitPrice")
oi["TotalPrice"] = oi.TotalPrice.fillna(oi.Quantity * oi.UnitPrice)
keep = oi.OrderID.isin(C["orders"].OrderID) & oi.FoodItemID.isin(C["menu"].FoodItemID); note("order_items", "orphan FK", "OrderID,FoodItemID", (~keep).sum(), "dropped"); C["order_items"] = oi[keep]

p = to_num(raw["payments"].copy().drop_duplicates(), ["PaymentID", "OrderID"])
for c in ["PaymentMethod", "PaymentStatus", "TransactionID"]: p[c] = clean_text(p[c])
p["PaymentDate"], _, _ = parse_mixed_dates(p.PaymentDate)
rank = p.PaymentStatus.map({"Success": 0, "Refunded": 1, "Pending": 2, "Failed": 3}).fillna(4)        # one payment per order: best status first
p = p.assign(_r=rank).sort_values(["OrderID", "_r", "PaymentID"], ascending=[True, True, False])
note("payments", "multiple payments per order", "OrderID", p.duplicated("OrderID").sum(), "kept best-status (Success first) row")
p = p.drop_duplicates("OrderID").drop(columns="_r").sort_values("PaymentID"); keep = p.OrderID.isin(C["orders"].OrderID)
note("payments", "orphan FK", "OrderID", (~keep).sum(), "dropped"); C["payments"] = p[keep]

f = raw["customer_feedback"].copy(); note("customer_feedback", "duplicate rows", "OrderID", f.duplicated("OrderID").sum(), "kept first feedback per order")
f = to_num(f.drop_duplicates().drop_duplicates("OrderID"), ["FeedbackID", "OrderID", "CustomerRating", "DeliveryRating", "FoodRating"])
for c in ["CustomerRating", "DeliveryRating", "FoodRating"]:
    bad = f[c] > 5; note("customer_feedback", "rating out of 1-5", c, bad.sum(), "set NaN then imputed"); f.loc[bad, c] = np.nan
for c, others in [("CustomerRating", ["DeliveryRating", "FoodRating"]), ("DeliveryRating", ["CustomerRating", "FoodRating"]), ("FoodRating", ["CustomerRating", "DeliveryRating"])]:
    note("customer_feedback", "missing", c, f[c].isna().sum(), "imputed with mean of other two ratings"); f[c] = f[c].fillna(f[others].mean(axis=1)).fillna(f[c].median()).round()
f["Review"], f["Sentiment"] = clean_text(f.Review), clean_text(f.Sentiment); keep = f.OrderID.isin(C["orders"].OrderID)
note("customer_feedback", "orphan FK", "OrderID", (~keep).sum(), "dropped"); C["customer_feedback"] = f[keep].astype({c: int for c in ["CustomerRating", "DeliveryRating", "FoodRating"]})
print({t: d.shape for t, d in C.items()})

{'cities': (25, 5), 'customers': (12000, 13), 'restaurants': (1200, 13), 'menu': (8997, 8), 'delivery_partners': (2000, 10), 'promotions': (300, 6), 'weather': (18264, 7), 'traffic': (18335, 6), 'orders': (20475, 15), 'order_items': (42361, 6), 'payments': (20070, 6), 'customer_feedback': (14065, 7)}


## 4. Validate

In [13]:
# ---- Validation: primary keys, foreign keys, ranges --------------------------------------------------------
pk = {"cities": "CityID", "customers": "CustomerID", "restaurants": "RestaurantID", "menu": "FoodItemID", "delivery_partners": "DeliveryPartnerID", "promotions": "PromotionID",
      "orders": "OrderID", "order_items": "OrderItemID", "payments": "PaymentID", "customer_feedback": "FeedbackID", "weather": "WeatherID", "traffic": "TrafficID"}
checks = [(f"{t}.{k} is unique", bool(C[t][k].is_unique)) for t, k in pk.items()]
fks = [("customers", "City", "cities", "City"), ("restaurants", "City", "cities", "City"), ("delivery_partners", "City", "cities", "City"), ("weather", "City", "cities", "City"),
       ("traffic", "City", "cities", "City"), ("menu", "RestaurantID", "restaurants", "RestaurantID"), ("orders", "CustomerID", "customers", "CustomerID"),
       ("orders", "RestaurantID", "restaurants", "RestaurantID"), ("orders", "DeliveryPartnerID", "delivery_partners", "DeliveryPartnerID"),
       ("order_items", "OrderID", "orders", "OrderID"), ("order_items", "FoodItemID", "menu", "FoodItemID"), ("payments", "OrderID", "orders", "OrderID"), ("customer_feedback", "OrderID", "orders", "OrderID")]
checks += [(f"FK {a}.{b} -> {c}.{e}", bool(C[a][b].dropna().isin(C[c][e]).all())) for a, b, c, e in fks]
o = C["orders"]
checks += [("orders: no negative FoodCost", bool((o.FoodCost >= 0).all())), ("orders: FinalAmount has no nulls", bool(o.FinalAmount.notna().all())),
           ("orders: DeliveryTime in (0,180]", bool(o.DeliveryTimeMinutes.dropna().between(0, 180).all())), ("restaurants: Rating in 1-5", bool(C["restaurants"].Rating.between(1, 5).all())),
           ("customers: Age in 10-100", bool(C["customers"].Age.between(10, 100).all())), ("promotions: End >= Start", bool((C["promotions"].EndDate >= C["promotions"].StartDate).all()))]
val = pd.DataFrame(checks, columns=["check", "passed"]); print(f"{int(val.passed.sum())}/{len(val)} checks passed"); val[~val.passed]

31/31 checks passed


,check,passed


In [14]:
# ---- The cleaning log -------------------------------------------------------------------------------------
cl = pd.DataFrame(log); print(len(cl), "logged transformations")
cl.groupby("table").agg(entries=("issue", "count"), rows_affected=("rows_affected", "sum")).sort_values("rows_affected", ascending=False)

52 logged transformations


,entries,rows_affected
table,,
orders,8,6005
customers,12,5237
weather,6,3673
traffic,4,3314
order_items,3,869
customer_feedback,5,681
payments,2,423
delivery_partners,3,242
menu,2,222


## 5. Save

In [15]:
# ---- Save cleaned tables (integers written WITHOUT a trailing .0 so PostgreSQL \\copy works) --------------------
INT_COLS = {"cities": ["CityID", "Population"], "customers": ["CustomerID", "Age", "TotalOrders"], "restaurants": ["RestaurantID"],
            "menu": ["FoodItemID", "RestaurantID", "PreparationTime", "Calories"], "delivery_partners": ["DeliveryPartnerID", "Age", "CompletedDeliveries"],
            "promotions": ["PromotionID", "DiscountPercentage"], "orders": ["OrderID", "CustomerID", "RestaurantID", "DeliveryPartnerID", "DeliveryTimeMinutes"],
            "order_items": ["OrderItemID", "OrderID", "FoodItemID", "Quantity"], "payments": ["PaymentID", "OrderID"],
            "customer_feedback": ["FeedbackID", "OrderID", "CustomerRating", "DeliveryRating", "FoodRating"], "weather": ["WeatherID", "Humidity"], "traffic": ["TrafficID"]}
for t, df in C.items():
    out = df.copy()
    for c in INT_COLS.get(t, []): out[c] = pd.to_numeric(out[c], errors="coerce").round().astype("Int64")
    out.to_csv(CLEAN / f"{t}.csv", index=False)
cl.to_csv(CLEAN / "cleaning_log.csv", index=False); val.to_csv(CLEAN / "validation_report.csv", index=False)
print("saved", len(C), "cleaned tables + cleaning_log.csv + validation_report.csv to", CLEAN)

saved 12 cleaned tables + cleaning_log.csv + validation_report.csv to /sessions/loving-awesome-archimedes/mnt/internship/Zomato_Business_Intelligence_Project/data/cleaned


## 6. Order-level analytical base table\nJoins only (no engineered features yet - those are notebook 03).

In [16]:
# ---- Analytical base table (ABT), step 1: join the cleaned tables at ORDER level (PRD week 2, day 8) ------------------
o = C["orders"].copy()
o["OrderTS"] = pd.to_datetime(o.OrderDate.dt.strftime("%Y-%m-%d") + " " + o.OrderTime)
o["order_min"] = o.OrderTS.dt.hour * 60 + o.OrderTS.dt.minute

r = C["restaurants"].rename(columns={"City": "RCity", "Rating": "RestaurantRating", "Area": "RArea"})
base = o.merge(r[["RestaurantID", "RestaurantName", "RCity", "RArea", "Cuisine", "RestaurantRating", "AverageCost", "RestaurantType", "Latitude", "Longitude"]], on="RestaurantID")
cu = C["customers"].rename(columns={"City": "CCity", "Age": "CustomerAge"})
base = base.merge(cu[["CustomerID", "CCity", "CustomerAge", "Membership", "RegistrationDate", "Gender"]], on="CustomerID")
dp = C["delivery_partners"].rename(columns={"Rating": "PartnerRating", "City": "PCity"})
base = base.merge(dp[["DeliveryPartnerID", "VehicleType", "PartnerRating", "AverageDeliveryTime", "CompletedDeliveries", "PCity"]], on="DeliveryPartnerID")

# weather: dates repeat within a city -> average per (city, date) first
w = C["weather"].groupby(["City", "Date"], as_index=False).agg(Rainfall=("Rainfall", "mean"), Temperature=("Temperature", "mean"), Humidity=("Humidity", "mean"),
                                                             WeatherCondition=("WeatherCondition", lambda s: s.mode().iat[0]))
base = base.merge(w, left_on=["RCity", "OrderDate"], right_on=["City", "Date"], how="left").drop(columns=["City", "Date"])
base["HasWeather"] = base.Rainfall.notna().astype(int)

# traffic: of the readings for that city+date, keep the one closest in time of day
tr = C["traffic"].copy(); hm = pd.to_datetime(tr.Time, format="%H:%M", errors="coerce"); tr["t_min"] = hm.dt.hour * 60 + hm.dt.minute
tr["TrafficScore"] = tr.TrafficLevel.map({"Low": 1, "Moderate": 2, "High": 3, "Severe": 4})
m = base[["OrderID", "RCity", "OrderDate", "order_min"]].merge(tr[["City", "Date", "t_min", "TrafficScore", "AverageSpeed"]], left_on=["RCity", "OrderDate"], right_on=["City", "Date"])
m["gap"] = (m.order_min - m.t_min).abs(); m = m.sort_values(["OrderID", "gap"]).drop_duplicates("OrderID")
base = base.merge(m[["OrderID", "TrafficScore", "AverageSpeed"]], on="OrderID", how="left"); base["HasTraffic"] = base.TrafficScore.notna().astype(int)
for c in ["Rainfall", "Temperature", "Humidity", "TrafficScore", "AverageSpeed"]:          # unmatched city-days -> city median (flagged above)
    base[c] = base[c].fillna(base.groupby("RCity")[c].transform("median")).fillna(base[c].median())
base["WeatherCondition"] = base.WeatherCondition.fillna("Unknown")

oi = C["order_items"].groupby("OrderID").agg(BasketSize=("Quantity", "sum"), ItemLines=("OrderItemID", "count")); base = base.merge(oi, on="OrderID", how="left")
base[["BasketSize", "ItemLines"]] = base[["BasketSize", "ItemLines"]].fillna(0)
fb = C["customer_feedback"].assign(AvgRating=lambda x: x[["CustomerRating", "DeliveryRating", "FoodRating"]].mean(axis=1))
base = base.merge(fb[["OrderID", "CustomerRating", "DeliveryRating", "FoodRating", "AvgRating", "Sentiment"]], on="OrderID", how="left")
base = base.merge(C["payments"][["OrderID", "PaymentStatus"]], on="OrderID", how="left").sort_values("OrderID").reset_index(drop=True)
print(base.shape, f"| weather matched {base.HasWeather.mean():.1%} | traffic matched {base.HasTraffic.mean():.1%}")
base.to_csv(CLEAN / "order_base.csv", index=False); base.head(3)

(20475, 52) | weather matched 63.0% | traffic matched 63.6%


,OrderID,CustomerID,RestaurantID,DeliveryPartnerID,OrderDate,OrderTime,DeliveryTimeMinutes,FoodCost,DeliveryFee,Discount,CouponCode,GST,FinalAmount,OrderStatus,PaymentMethod,OrderTS,order_min,RestaurantName,RCity,RArea,Cuisine,RestaurantRating,AverageCost,RestaurantType,Latitude,Longitude,CCity,CustomerAge,Membership,RegistrationDate,Gender,VehicleType,PartnerRating,AverageDeliveryTime,CompletedDeliveries,PCity,Rainfall,Temperature,Humidity,WeatherCondition,HasWeather,TrafficScore,AverageSpeed,HasTraffic,BasketSize,ItemLines,CustomerRating,DeliveryRating,FoodRating,AvgRating,Sentiment,PaymentStatus
0,100001,7095,474,121,2023-06-17,11:51:00,43.0,206,40,103.0,ZOMKXZZW,10.30,153.30,Delivered,Cash on Delivery,2023-06-17 11:51:00,711,Urban Treats,Bengaluru,Market Area,Mexican,3.9,432,Dine-in & Delivery,25.971353,74.945306,Chennai,32,Basic,2024-06-19,Male,Scooter,4.0,47.3,334,Raipur,8.80,26.9,59.5,Unknown,0,2.0,23.4,1,4,2,2.0,2.0,1.0,1.666667,Negative,Success
1,100002,9498,828,1776,2023-07-01,10:18:00,18.0,425,20,127.5,ZOM72K4N,21.25,338.75,Food Not Delivered,Debit Card,2023-07-01 10:18:00,618,Curry Bites,Lucknow,MG Road,Sushi,4.5,351,Cloud Kitchen,26.969519,76.641600,Chennai,32,Zomato Pro,2022-10-17,Male,Bike,4.2,47.7,554,Ludhiana,328.65,30.7,80.0,Rainy,1,2.0,23.0,0,4,2,4.0,3.0,4.0,3.666667,Positive,Success
2,100003,11040,105,27,2023-05-09,19:34:00,50.0,287,30,28.7,ZOMQJT3W,14.35,302.65,Cancelled,Net Banking,2023-05-09 19:34:00,1174,Royal Grill,Delhi,Central,Unknown,3.8,682,QSR,30.433987,84.935762,Guwahati,53,Basic,2024-05-31,Other,Bike,3.8,29.6,217,Vadodara,1.80,34.2,68.0,Cloudy,1,2.0,22.6,0,1,1,NaN,NaN,NaN,NaN,<NA>,Refunded


## Limitations carried forward\n* ~15% of slash dates are ambiguous; orders use the payment date to disambiguate, weather/traffic default to DD/MM.\n* Weather/traffic dates repeat (about 63% of city-days exist), so only ~63% of orders link to them; unmatched values are imputed with the city median and flagged (`HasWeather`, `HasTraffic`).\n* Customer city and restaurant city agree for only ~4% of orders and customers have no coordinates, so true delivery distance cannot be computed.